#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행 
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.


import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import *




* 함수 생성

In [2]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [3]:
data_file = r'../데이터/data01_train.csv'
data = pd.read_csv(data_file)
data.drop('subject',axis=1,inplace= True)
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS


#### 2) 기본 정보 조회

In [4]:
#전체 데이터의 행,열 개수 확인
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 562 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), object(1)
memory usage: 25.2+ MB


In [5]:
#전체 데이터의 상위 5개 행 확인
data.head(5)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS


In [6]:
#전체 데이터의 수치형 변수 분포 확인
data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,0.126955,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.249176,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.965725,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.021610,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,0.133887,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,0.288944,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.946700,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000


In [7]:
#전체 데이터의 모든 변수 확인
data.columns

Index(['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
       'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X',
       ...
       'fBodyBodyGyroJerkMag-skewness()', 'fBodyBodyGyroJerkMag-kurtosis()',
       'angle(tBodyAccMean,gravity)', 'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)', 'Activity'],
      dtype='object', length=562)

## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다. 


### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [8]:
# 데이터 분할을 위한 전처리
target = 'Activity'
x = data.drop(target, axis = 1)
y = data.loc[:, target]

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size = .3)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [9]:
from sklearn.preprocessing import StandardScaler

# 특성과 레이블 나누기
X = data.drop('Activity', axis=1)
y = data['Activity']

# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)  # ← DataFrame으로 다시 변환

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [10]:
from sklearn.model_selection import train_test_split

# 특성 (X_scaled)과 레이블 (y)을 사용
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled_df, y, 
    test_size=0.2,          # ← 8:2 기준
    random_state=42,        # ← 재현성 보장
    stratify=y              # ← 클래스 불균형 고려 (추천)
)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다. 
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

In [13]:
# 상대 경로로 변수 중요도 CSV 불러오기
#feature_importance_df = pd.read_csv('../../2024_aix_final/fi_analysis_with_all_importances.csv')
#"C:\Users\syj04\Code\2025-1AIX\데이터\fi_analysis_ver2.csv"
feature_importance_df = pd.read_csv("C:\\Users\\syj04\\Code\\2025-1AIX\\데이터\\fi_analysis_ver2.csv")

In [14]:
# 상위 N개 feature만 뽑기 위한 함수
def select_top_n_features(X, feature_importance_df, n):
    top_features = feature_importance_df['feature_name'].iloc[:n].values
    return X[top_features]

In [15]:
# 상위 N개 선택
n = 20
X_train_top = select_top_n_features(X_train, feature_importance_df, n)
X_val_top = select_top_n_features(X_val, feature_importance_df, n)

### (1) 알고리즘1 : 로지스틱 회귀

In [17]:
# 전체 feature 사용
algo1 = LogisticRegression(max_iter=1000, random_state=42)
algo1.fit(X_train, y_train)
pred1 = algo1.predict(X_val)

acc1 = accuracy_score(y_val, pred1)
print("알고리즘1 (Logistic Regression) 전체 정확도:", acc1)

알고리즘1 (Logistic Regression) 전체 정확도: 0.983857264231096


In [18]:
# 상위 N개 feature 사용
algo1_top = LogisticRegression(max_iter=1000, random_state=42)
algo1_top.fit(X_train_top, y_train)
pred1_top = algo1_top.predict(X_val_top)
acc1_top = accuracy_score(y_val, pred1_top)
print("알고리즘1 (Logistic Regression) 상위 N개 정확도:", acc1_top)

알고리즘1 (Logistic Regression) 상위 N개 정확도: 0.7068819031435853


### (2) 알고리즘2 : 랜덤 포레스트

In [19]:
from sklearn.ensemble import RandomForestClassifier

# 전체 feature 사용
algo2 = RandomForestClassifier(random_state=42)
algo2.fit(X_train, y_train)
pred2 = algo2.predict(X_val)
acc2 = accuracy_score(y_val, pred2)
print("알고리즘2 (Random Forest) 전체 정확도:", acc2)

알고리즘2 (Random Forest) 전체 정확도: 0.9813084112149533


In [20]:
# 상위 N개 feature 사용
algo2_top = RandomForestClassifier(random_state=42)
algo2_top.fit(X_train_top, y_train)
pred2_top = algo2_top.predict(X_val_top)
acc2_top = accuracy_score(y_val, pred2_top)
print("알고리즘2 (Random Forest) 상위 N개 정확도:", acc2_top)

알고리즘2 (Random Forest) 상위 N개 정확도: 0.7952421410365336


### (3) 알고리즘3 : KNN

In [21]:
from sklearn.neighbors import KNeighborsClassifier

# 전체 feature 사용
algo3 = KNeighborsClassifier()
algo3.fit(X_train, y_train)
pred3 = algo3.predict(X_val)
acc3 = accuracy_score(y_val, pred3)
print("알고리즘3 (KNN) 전체 정확도:", acc3)

알고리즘3 (KNN) 전체 정확도: 0.9481733220050977


In [22]:
# 상위 N개 feature 사용
algo3_top = KNeighborsClassifier()
algo3_top.fit(X_train_top, y_train)
pred3_top = algo3_top.predict(X_val_top)
acc3_top = accuracy_score(y_val, pred3_top)
print("알고리즘3 (KNN) 상위 N개 정확도:", acc3_top)

알고리즘3 (KNN) 상위 N개 정확도: 0.7434154630416313


### (4) 알고리즘4 : XGboost

In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 레이블 인코딩
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# 전체 feature 사용
algo4 = XGBClassifier(random_state=42, eval_metric='mlogloss')
algo4.fit(X_train, y_train_encoded)
pred4 = algo4.predict(X_val)
acc4 = accuracy_score(y_val_encoded, pred4)
print("알고리즘4 (XGBoost) 전체 정확도:", acc4)

알고리즘4 (XGBoost) 전체 정확도: 0.9923534409515717


In [24]:
# 상위 N개 feature 사용
algo4_top = XGBClassifier(random_state=42, eval_metric='mlogloss')
algo4_top.fit(X_train_top, y_train_encoded)
pred4_top = algo4_top.predict(X_val_top)
acc4_top = accuracy_score(y_val_encoded, pred4_top)
print("알고리즘4 (XGBoost) 상위 N개 정확도:", acc4_top)


알고리즘4 (XGBoost) 상위 N개 정확도: 0.8181818181818182


### N = 50

In [25]:
# 상위 N개 선택
n = 50
X_train_top = select_top_n_features(X_train, feature_importance_df, n)
X_val_top = select_top_n_features(X_val, feature_importance_df, n)

#### 로지스틱 회귀

In [26]:
# 상위 N개 feature 사용
algo1_top = LogisticRegression(max_iter=1000, random_state=42)
algo1_top.fit(X_train_top, y_train)
pred1_top = algo1_top.predict(X_val_top)
acc1_top = accuracy_score(y_val, pred1_top)
print("알고리즘1 (Logistic Regression) 상위 N개 정확도:", acc1_top)

알고리즘1 (Logistic Regression) 상위 N개 정확도: 0.9422259983007647


#### 랜덤 포레스트

In [27]:
# 상위 N개 feature 사용
algo2_top = RandomForestClassifier(random_state=42)
algo2_top.fit(X_train_top, y_train)
pred2_top = algo2_top.predict(X_val_top)
acc2_top = accuracy_score(y_val, pred2_top)
print("알고리즘2 (Random Forest) 상위 N개 정확도:", acc2_top)

알고리즘2 (Random Forest) 상위 N개 정확도: 0.9634664401019541


#### KNN

In [28]:
# 상위 N개 feature 사용
algo3_top = KNeighborsClassifier()
algo3_top.fit(X_train_top, y_train)
pred3_top = algo3_top.predict(X_val_top)
acc3_top = accuracy_score(y_val, pred3_top)
print("알고리즘3 (KNN) 상위 N개 정확도:", acc3_top)

알고리즘3 (KNN) 상위 N개 정확도: 0.8912489379779099


### XGBoost

In [29]:
# 상위 N개 feature 사용
algo4_top = XGBClassifier(random_state=42, eval_metric='mlogloss')
algo4_top.fit(X_train_top, y_train_encoded)
pred4_top = algo4_top.predict(X_val_top)
acc4_top = accuracy_score(y_val_encoded, pred4_top)
print("알고리즘4 (XGBoost) 상위 N개 정확도:", acc4_top)

알고리즘4 (XGBoost) 상위 N개 정확도: 0.9821580288870009


### 알고리즘별 정확도 비교 표

In [30]:
import pandas as pd

# 알고리즘별 정확도 정리
accuracy_results = {
    '로지스틱회귀': {
        '전체 변수 정확도': acc1,
        '상위 N개 변수 정확도': acc1_top
    },
    '랜덤포레스트': {
        '전체 변수 정확도': acc2,
        '상위 N개 변수 정확도': acc2_top
    },
    'KNN': {
        '전체 변수 정확도': acc3,
        '상위 N개 변수 정확도': acc3_top
    },
    'XGBoost': {
        '전체 변수 정확도': acc4,
        '상위 N개 변수 정확도': acc4_top
    }
}

# 데이터프레임으로 변환
acc_df = pd.DataFrame.from_dict(accuracy_results, orient='index')

# 소수점 4자리로 보기 좋게
acc_df = acc_df.round(4)

# 출력
acc_df


,전체 변수 정확도,상위 N개 변수 정확도
로지스틱회귀,0.9839,0.9422
랜덤포레스트,0.9813,0.9635
KNN,0.9482,0.8912
XGBoost,0.9924,0.9822
